In [0]:
%run /Workspace/Repos/pacioianu4@gmail.com/spotify-end-to-end-api-project/src/utils/get_auth_and_refresh_token

In [0]:
import json
import uuid
from datetime import datetime, timezone
from urllib.parse import urlparse, parse_qs

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("raw_base_path","/Workspace/Users/pacioianu4@gmail.com/Files/spotify-end-to-end-api-project/data/raw","RAW base path")
dbutils.widgets.text("scope_name", "spotify_secrets", "Secret scope name")
dbutils.widgets.text("client_id_key", "client_id", "Secret key: client_id")
dbutils.widgets.text("client_secret_key", "client_secret", "Secret key: client_secret")
dbutils.widgets.text("refresh_token_key", "refresh_token", "Secret key: refresh-token")

dbutils.widgets.text("limit", "50", "Pagination limit (max 50)")
dbutils.widgets.text("max_pages", "200", "Max pages safeguard")
dbutils.widgets.text("max_playlists", "2000", "Max playlists safeguard")
dbutils.widgets.text("max_playlist_tracks_pages", "4000", "Max playlist tracks pages safeguard")

dbutils.widgets.text("enable_me", "true", "Ingest /v1/me")
dbutils.widgets.text("enable_me_playlists", "true", "Ingest /v1/me/playlists")
dbutils.widgets.text("enable_playlist_tracks", "true", "Ingest /v1/playlists/{id}/tracks")
dbutils.widgets.text("enable_catalog_enrich", "true", "Ingest catalog enrich (tracks/artists/audio-features)")
dbutils.widgets.text("catalog_enrich_track_ids_limit", "2000", "Max unique track_ids to enrich per run")

RAW_BASE_PATH = dbutils.widgets.get("raw_base_path").rstrip("/")
SCOPE_NAME = dbutils.widgets.get("scope_name")
CLIENT_ID_KEY = dbutils.widgets.get("client_id_key")
CLIENT_SECRET_KEY = dbutils.widgets.get("client_secret_key")
REFRESH_TOKEN_KEY = dbutils.widgets.get("refresh_token_key")

LIMIT = int(dbutils.widgets.get("limit"))
MAX_PAGES = int(dbutils.widgets.get("max_pages"))
MAX_PLAYLISTS = int(dbutils.widgets.get("max_playlists"))
MAX_PLAYLIST_TRACKS_PAGES = int(dbutils.widgets.get("max_playlist_tracks_pages"))

ENABLE_ME = dbutils.widgets.get("enable_me").lower() == "true"
ENABLE_ME_PLAYLISTS = dbutils.widgets.get("enable_me_playlists").lower() == "true"
ENABLE_PLAYLIST_TRACKS = dbutils.widgets.get("enable_playlist_tracks").lower() == "true"
ENABLE_CATALOG_ENRICH = dbutils.widgets.get("enable_catalog_enrich").lower() == "true"
CATALOG_TRACK_IDS_LIMIT = int(dbutils.widgets.get("catalog_enrich_track_ids_limit"))


In [0]:
client_id = dbutils.secrets.get(SCOPE_NAME, CLIENT_ID_KEY)
client_secret = dbutils.secrets.get(SCOPE_NAME, CLIENT_SECRET_KEY)
refresh_token_fallback = dbutils.secrets.get(SCOPE_NAME, REFRESH_TOKEN_KEY)

In [0]:
def utc_now():
    return datetime.now(timezone.utc)

def safe_name(s: str) -> str:
    return "".join(c if c.isalnum() or c in ("-", "_", "=") else "_" for c in s)

def ensure_dir(path: str):
    dbutils.fs.mkdirs(path)

def write_json(path: str, obj):
    dbutils.fs.put(path, json.dumps(obj, ensure_ascii=False), overwrite=True)

def build_run_dir(raw_base: str, entity: str, run_id: str) -> str:
    d = utc_now().strftime("%Y-%m-%d")
    return f"{raw_base}/{safe_name(entity)}/ingestion_date={d}/run_id={safe_name(run_id)}"

def save_page_json(run_dir: str, page_idx: int, request_url: str, params: dict | None, payload: dict, extra_meta: dict | None = None):
    page_path = f"{run_dir}/page_{page_idx:04d}.json"
    meta_path = f"{run_dir}/page_{page_idx:04d}_meta.json"
    write_json(page_path, payload)
    meta = {
        "ts_utc": utc_now().isoformat(),
        "page_idx": page_idx,
        "request_url": request_url,
        "request_params": params or {},
        "payload_top_keys": list(payload.keys()) if isinstance(payload, dict) else None
    }
    if extra_meta:
        meta.update(extra_meta)
    write_json(meta_path, meta)

def chunk_list(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def api_get(url: str, params: dict | None = None, run_id: str | None = None) -> dict:
    return spotify_api_call(
        method="GET",
        url=url,
        client_id=client_id,
        client_secret=client_secret,
        refresh_token_fallback=refresh_token_fallback,
        params=params,
        json_body=None,     # ✅
        timeout_s=30,
        run_id=run_id
    )

In [0]:
def ingest_me(run_id: str) -> dict:
    entity = "me"
    run_dir = build_run_dir(RAW_BASE_PATH, entity, run_id)
    ensure_dir(run_dir)
    write_json(f"{run_dir}/_meta.json", {"ts_utc": utc_now().isoformat(), "entity": entity, "run_id": run_id})

    payload = api_get("https://api.spotify.com/v1/me")
    save_page_json(run_dir, 1, "https://api.spotify.com/v1/me", None, payload)
    return {"entity": entity, "run_dir": run_dir, "pages": 1}


In [0]:
def ingest_me_playlists(run_id: str) -> tuple[dict, list]:
    entity = "me_playlists"
    run_dir = build_run_dir(RAW_BASE_PATH, entity, run_id)
    ensure_dir(run_dir)
    write_json(f"{run_dir}/_meta.json", {
        "ts_utc": utc_now().isoformat(),
        "entity": entity,
        "run_id": run_id,
        "start_url": "https://api.spotify.com/v1/me/playlists",
        "limit": LIMIT
    })

    url = "https://api.spotify.com/v1/me/playlists"
    params = {"limit": LIMIT, "offset": 0}

    playlists = []
    page_idx = 1
    next_url = url

    while next_url and page_idx <= MAX_PAGES and len(playlists) < MAX_PLAYLISTS:
        payload = api_get(next_url, params if page_idx == 1 else None)
        save_page_json(run_dir, page_idx, next_url, params if page_idx == 1 else None, payload)

        items = payload.get("items", []) if isinstance(payload, dict) else []
        for p in items:
            pid = p.get("id")
            if pid:
                playlists.append(pid)
                if len(playlists) >= MAX_PLAYLISTS:
                    break

        next_url = payload.get("next")
        page_idx += 1
        params = None  # după prima pagină, next URL include querystring

    return ({"entity": entity, "run_dir": run_dir, "pages": page_idx - 1, "playlists_found": len(playlists)}, playlists)

In [0]:
def ingest_playlist_tracks(run_id: str, playlist_ids: list[str]) -> tuple[dict, list[str], list[str]]:
    entity = "playlist_tracks"
    run_dir = build_run_dir(RAW_BASE_PATH, entity, run_id)
    ensure_dir(run_dir)
    write_json(f"{run_dir}/_meta.json", {
        "ts_utc": utc_now().isoformat(),
        "entity": entity,
        "run_id": run_id,
        "limit": LIMIT,
        "playlists_count": len(playlist_ids)
    })

    track_ids = []
    artist_ids = []
    page_idx_global = 1
    pages_total = 0

    for idx, playlist_id in enumerate(playlist_ids, start=1):
        if page_idx_global > MAX_PLAYLIST_TRACKS_PAGES:
            break

        base_url = f"https://api.spotify.com/v1/playlists/{playlist_id}/tracks"
        params = {"limit": LIMIT, "offset": 0}
        next_url = base_url
        page_idx_local = 1

        while next_url and page_idx_global <= MAX_PLAYLIST_TRACKS_PAGES:
            payload = api_get(next_url, params if page_idx_local == 1 else None)

            # include playlist_id în meta ca să poți filtra ușor
            save_page_json(
                run_dir,
                page_idx_global,
                next_url,
                params if page_idx_local == 1 else None,
                payload,
                extra_meta={"playlist_id": playlist_id, "playlist_idx": idx, "page_idx_local": page_idx_local}
            )

            items = payload.get("items", []) if isinstance(payload, dict) else []
            for it in items:
                tr = (it or {}).get("track") or {}
                tid = tr.get("id")
                if tid:
                    track_ids.append(tid)

                for a in tr.get("artists", []) or []:
                    aid = (a or {}).get("id")
                    if aid:
                        artist_ids.append(aid)

            next_url = payload.get("next")
            params = None
            page_idx_local += 1
            page_idx_global += 1
            pages_total += 1

    # dedup + limit pentru enrich
    track_ids_unique = list(dict.fromkeys([t for t in track_ids if t]))[:CATALOG_TRACK_IDS_LIMIT]
    artist_ids_unique = list(dict.fromkeys([a for a in artist_ids if a]))  # artist count de obicei mai mic

    return (
        {"entity": entity, "run_dir": run_dir, "pages": pages_total, "tracks_unique": len(track_ids_unique), "artists_unique": len(artist_ids_unique)},
        track_ids_unique,
        artist_ids_unique
    )

In [0]:
def ingest_tracks_bulk(run_id: str, track_ids: list[str]) -> dict:
    entity = "tracks_bulk"
    run_dir = build_run_dir(RAW_BASE_PATH, entity, run_id)
    ensure_dir(run_dir)
    write_json(f"{run_dir}/_meta.json", {"ts_utc": utc_now().isoformat(), "entity": entity, "run_id": run_id, "ids_count": len(track_ids)})

    page_idx = 1
    for chunk in chunk_list(track_ids, 50):
        ids = ",".join(chunk)
        url = "https://api.spotify.com/v1/tracks"
        payload = api_get(url, {"ids": ids})
        save_page_json(run_dir, page_idx, url, {"ids": f"<{len(chunk)} ids>"}, payload, extra_meta={"ids_sample": chunk[:5]})
        page_idx += 1

    return {"entity": entity, "run_dir": run_dir, "pages": page_idx - 1}


In [0]:
def ingest_artists_bulk(run_id: str, artist_ids: list[str]) -> dict:
    entity = "artists_bulk"
    run_dir = build_run_dir(RAW_BASE_PATH, entity, run_id)
    ensure_dir(run_dir)
    write_json(f"{run_dir}/_meta.json", {"ts_utc": utc_now().isoformat(), "entity": entity, "run_id": run_id, "ids_count": len(artist_ids)})

    page_idx = 1
    for chunk in chunk_list(artist_ids, 50):
        ids = ",".join(chunk)
        url = "https://api.spotify.com/v1/artists"
        payload = api_get(url, {"ids": ids})
        save_page_json(run_dir, page_idx, url, {"ids": f"<{len(chunk)} ids>"}, payload, extra_meta={"ids_sample": chunk[:5]})
        page_idx += 1

    return {"entity": entity, "run_dir": run_dir, "pages": page_idx - 1}

In [0]:
def ingest_audio_features_bulk(run_id: str, track_ids: list[str]) -> dict:
    entity = "audio_features_bulk"
    run_dir = build_run_dir(RAW_BASE_PATH, entity, run_id)
    ensure_dir(run_dir)
    write_json(f"{run_dir}/_meta.json", {"ts_utc": utc_now().isoformat(), "entity": entity, "run_id": run_id, "ids_count": len(track_ids)})

    page_idx = 1
    for chunk in chunk_list(track_ids, 100):
        ids = ",".join(chunk)
        url = "https://api.spotify.com/v1/audio-features"
        payload = api_get(url, {"ids": ids})
        save_page_json(run_dir, page_idx, url, {"ids": f"<{len(chunk)} ids>"}, payload, extra_meta={"ids_sample": chunk[:5]})
        page_idx += 1

    return {"entity": entity, "run_dir": run_dir, "pages": page_idx - 1}

In [0]:
master_run_id = str(uuid.uuid4())

run_summary = {
    "master_run_id": master_run_id,
    "ts_utc": utc_now().isoformat(),
    "raw_base_path": RAW_BASE_PATH,
    "steps": []
}

# 1) /v1/me
if ENABLE_ME:
    run_summary["steps"].append(ingest_me(master_run_id))

# 2) /v1/me/playlists
playlist_ids = []
if ENABLE_ME_PLAYLISTS:
    step, playlist_ids = ingest_me_playlists(master_run_id)
    run_summary["steps"].append(step)

# 3) /v1/playlists/{id}/tracks
track_ids = []
artist_ids = []
if ENABLE_PLAYLIST_TRACKS and playlist_ids:
    step, track_ids, artist_ids = ingest_playlist_tracks(master_run_id, playlist_ids)
    run_summary["steps"].append(step)

# 4) catalog enrich
if ENABLE_CATALOG_ENRICH and track_ids:
    run_summary["steps"].append(ingest_tracks_bulk(master_run_id, track_ids))
    run_summary["steps"].append(ingest_audio_features_bulk(master_run_id, track_ids))

    # dedup artists (din playlist tracks) și limitează rezonabil (ex: 5000)
    artist_ids = list(dict.fromkeys(artist_ids))[:5000]
    if artist_ids:
        run_summary["steps"].append(ingest_artists_bulk(master_run_id, artist_ids))

# write master meta
master_entity = "run_manifest"
manifest_dir = build_run_dir(RAW_BASE_PATH, master_entity, master_run_id)
ensure_dir(manifest_dir)
write_json(f"{manifest_dir}/manifest.json", run_summary)

run_summary